In [2]:
!pip install openai-whisper librosa transformers torch --quiet
!apt-get install -y ffmpeg --quiet

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 4.9 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [3]:
import whisper
import librosa
import numpy as np
import matplotlib.pyplot as plt
from transformers import pipeline

whisper_model = whisper.load_model("base")
'''sentiment = pipeline("text-classification",
                     model="cardiffnlp/twitter-roberta-base-sentiment-latest",
                     top_k=1)''' # Had to replace with other model due to language issue

100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 93.1MiB/s]


'sentiment = pipeline("text-classification",\n                     model="cardiffnlp/twitter-roberta-base-sentiment-latest",\n                     top_k=1)'

In [4]:
sentiment = pipeline("text-classification",
                     model="tabularisai/multilingual-sentiment-analysis",
                     top_k=1)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/851 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [5]:
def get_audio_mood(path):
    y, sr = librosa.load(path, duration=60)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    energy = np.mean(librosa.feature.rms(y=y))
    brightness = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    tempo_val = float(tempo) if np.isscalar(tempo) else float(tempo.item())
    return round((min(tempo_val/200,1) + min(float(energy)/0.1,1) + min(float(brightness)/4000,1)) / 3, 2)

def get_lyric_mood(path):
    text = whisper_model.transcribe(path)["text"][:400]
    result = sentiment(text, truncation=True, max_length=512)[0][0]
    label, score = result['label'].lower(), result['score']
    if 'positive' in label: return round(0.5 + score*0.5, 2)
    if 'negative' in label: return round(0.5 - score*0.5, 2)
    return 0.5

In [6]:
from google.colab import files
uploaded = files.upload()

Saving Channa Mereya - Ae Dil Hai Mushkil (320 kbps).mp3 to Channa Mereya - Ae Dil Hai Mushkil (320 kbps).mp3
Saving Lovely - Billie Ellish.mp3 to Lovely - Billie Ellish.mp3
Saving Ramba Ho - Dhurandhar (320 kbps).mp3 to Ramba Ho - Dhurandhar (320 kbps).mp3
Saving Saiyaara - Saiyaara (320 kbps).mp3 to Saiyaara - Saiyaara (320 kbps).mp3
Saving Sanam Re - Sanam Re (320 kbps).mp3 to Sanam Re - Sanam Re (320 kbps).mp3
Saving Sandese Aate Hain Border 320 Kbps.mp3 to Sandese Aate Hain Border 320 Kbps.mp3
Saving Tum Hi Ho - Aashiqui 2 (320 kbps).mp3 to Tum Hi Ho - Aashiqui 2 (320 kbps).mp3


In [7]:
results = []
for path in uploaded:
    name = path.rsplit('.',1)[0].replace('_',' ')
    am = get_audio_mood(path)
    lm = get_lyric_mood(path)
    results.append((name, am, lm, round(abs(am-lm), 2)))
    print(f"{name}: Audio={am}, Lyrics={lm}, Dissonance={round(abs(am-lm),2)}")

/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Channa Mereya - Ae Dil Hai Mushkil (320 kbps): audio=0.63, lyric=0.81, dissonance=0.18


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Lovely - Billie Ellish: audio=0.64, lyric=0.35, dissonance=0.29


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

Sanam Re - Sanam Re (320 kbps): audio=0.65, lyric=0.63, dissonance=0.02


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Sandese Aate Hain Border 320 Kbps: audio=0.69, lyric=0.74, dissonance=0.05


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Tum Hi Ho - Aashiqui 2 (320 kbps): audio=0.61, lyric=0.34, dissonance=0.27


In [ ]:
df = pd.DataFrame(results, columns=['Song', 'Audio', 'Lyrics', 'Dissonance'])
df.to_csv('output.csv', index=False)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

audio_scores = df['Audio'].tolist()
lyric_scores = df['Lyrics'].tolist()
names = df['Song'].tolist()

# quadrant backgrounds
ax.axhspan(0.5, 1.0, xmin=0, xmax=0.5, alpha=0.08, color='blue')    # sounds sad, feels happy
ax.axhspan(0.5, 1.0, xmin=0.5, xmax=1.0, alpha=0.08, color='green')  # sounds happy, feels happy
ax.axhspan(0.0, 0.5, xmin=0, xmax=0.5, alpha=0.08, color='gray')     # sounds sad, feels sad
ax.axhspan(0.0, 0.5, xmin=0.5, xmax=1.0, alpha=0.08, color='red')    # sounds happy, feels sad

# quadrant labels
ax.text(0.25, 0.75, 'sounds sad\nfeels happy', ha='center', va='center',
        fontsize=9, color='blue', alpha=0.6, transform=ax.transAxes)
ax.text(0.75, 0.75, 'sounds happy\nfeels happy', ha='center', va='center',
        fontsize=9, color='green', alpha=0.6, transform=ax.transAxes)
ax.text(0.25, 0.25, 'sounds sad\nfeels sad', ha='center', va='center',
        fontsize=9, color='gray', alpha=0.6, transform=ax.transAxes)
ax.text(0.75, 0.25, 'sounds happy\nfeels sad', ha='center', va='center',
        fontsize=9, color='red', alpha=0.6, transform=ax.transAxes)

# dividing lines
ax.axhline(0.5, color='black', linewidth=0.8, alpha=0.3)
ax.axvline(0.5, color='black', linewidth=0.8, alpha=0.3)

# scatter points
ax.scatter(audio_scores, lyric_scores, s=100, zorder=5,
           color='black', edgecolors='white', linewidths=0.5)

# song labels
for i, name in enumerate(names):
    ax.annotate(name, (audio_scores[i], lyric_scores[i]),
                xytext=(8, 4), textcoords='offset points', fontsize=8)

ax.set_xlabel('Music mood (Acoustics)', fontsize=11)
ax.set_ylabel('Lyric mood (Semantics)', fontsize=11)
ax.set_title('', fontsize=13, fontweight='normal')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('quadrant_plot_1.png', dpi=400)
plt.show()